In [1]:
class ExtendedEnv:
    def __init__(self):
        self.state = 0
        self.end_state = 5
        self.states = list(range(0, self.end_state + 1))  # tous les états
        self.actions = [0, 1]  # 0 = reculer, 1 = avancer

    def reset(self):
        self.state = 0
        return self.state

    def step(self, action):
        if action == 1 and self.state < self.end_state:
            self.state += 1
        elif action == 0 and self.state > 0:
            self.state -= 1

        reward = 1 if self.state == self.end_state else 0
        done = (self.state == self.end_state)
        return self.state, reward, done, {}

    def get_all_states(self):
        return self.states

    def get_all_actions(self):
        return self.actions


In [2]:
import random

def generate_episode(env, policy):
    episode = []
    state = env.reset()
    done = False

    while not done:
        actions = list(policy[state].keys())
        probs = list(policy[state].values())
        action = random.choices(actions, weights=probs, k=1)[0]  # TODO understand
        next_state, reward, done, _ = env.step(action)
        episode.append((state, action, reward))
        state = next_state

    return episode

In [ ]:
import numpy as np

def expected_sarsa_control(env, num_episodes, alpha=0.1, gamma=0.99, epsilon=0.1):
    all_states = env.get_all_states()
    all_actions = env.get_all_actions()

    # 🔹 Initialisation de Q(s, a)
    Q = {}
    for state in all_states:
        Q[state] = {action: 0.0 for action in all_actions}

    def get_policy_probs(state):
        """Politique epsilon-greedy sous forme de distribution de probabilité."""
        probs = {}
        best_action = max(Q[state], key=Q[state].get)
        for action in all_actions:
            probs[action] = epsilon / len(all_actions)
        probs[best_action] += 1.0 - epsilon
        return probs

    for episode_num in range(num_episodes):
        state = env.reset()
        done = False

        while not done:
            # 🔹 Choisir action selon epsilon-greedy
            action_probs = get_policy_probs(state)
            actions = list(action_probs.keys())
            probs = list(action_probs.values())
            action = np.random.choice(actions, p=probs)
            
            next_state, reward, done, _ = env.step(action)

            # 🔹 Calcul de l'espérance des Q(next_state, a') selon la politique
            next_action_probs = get_policy_probs(next_state)
            expected_q = sum([
                next_action_probs[a] * Q[next_state][a] for a in all_actions
            ])

            # 🔹 Mise à jour Expected SARSA
            Q[state][action] += alpha * (reward + gamma * expected_q - Q[state][action])

            state = next_state

    # 🔹 Politique finale greedy
    policy = {}
    for state in all_states:
        best_action = max(Q[state], key=Q[state].get)
        policy[state] = best_action

    return policy, Q


In [7]:
env = ExtendedEnv()
print(expected_sarsa_control(env, 10000))

TypeError: ExtendedEnv.step() takes 2 positional arguments but 3 were given